In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as sco
import os
import warnings

warnings.filterwarnings('ignore')

def otimizar_fronteira_eficiente(diretorio_dados, num_simulacoes=20000):
    print("=== INÍCIO DA OTIMIZAÇÃO DE MARKOWITZ (FRONTEIRA EFICIENTE) ===")
    
    # 1. Carregamento dos Parâmetros
    caminho_kpis = os.path.join(diretorio_dados, "kpis_ativos_markowitz.csv")
    caminho_cov = os.path.join(diretorio_dados, "matriz_covariancia_anualizada.csv")
    
    df_kpis = pd.read_csv(caminho_kpis, index_col=0)
    cov_matrix = pd.read_csv(caminho_cov, index_col=0)
    
    # Garantir que a ordem dos ativos é a mesma em ambas as matrizes
    ativos = df_kpis.index.tolist()
    cov_matrix = cov_matrix.loc[ativos, ativos]
    
    retornos_esperados = df_kpis['Retorno_Excesso_Anualizado'].values
    num_ativos = len(ativos)
    
    print(f"1. A processar {num_ativos} ativos para otimização...")
    
    # 2. Funções de Custo para o Otimizador
    def calcular_performance_portfolio(pesos, retornos, cov_matrix):
        retorno_port = np.sum(retornos * pesos)
        volatilidade_port = np.sqrt(np.dot(pesos.T, np.dot(cov_matrix, pesos)))
        return retorno_port, volatilidade_port
    
    def minimizar_sharpe_negativo(pesos, retornos, cov_matrix):
        ret, vol = calcular_performance_portfolio(pesos, retornos, cov_matrix)
        # O scipy só minimiza, então minimizamos o negativo do Sharpe para achar o Máximo Sharpe
        # Como já estamos usando prêmio de risco, Risk-Free Rate = 0
        return -(ret / vol)
    
    def minimizar_volatilidade(pesos, retornos, cov_matrix):
        return calcular_performance_portfolio(pesos, retornos, cov_matrix)[1]
    
    # Restrições (Soma dos pesos = 1) e Limites (Pesos entre 0 e 1, posição Long-Only)
    restricoes = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    limites = tuple((0, 1) for asset in range(num_ativos))
    
    # Peso inicial igualitário (1/N)
    pesos_iniciais = num_ativos * [1. / num_ativos,]
    
    # 3. Otimização Numérica (Scipy SLSQP)
    print("2. A otimizar o Portfólio de Máximo Índice de Sharpe...")
    otimizacao_sharpe = sco.minimize(minimizar_sharpe_negativo, pesos_iniciais, args=(retornos_esperados, cov_matrix),
                                     method='SLSQP', bounds=limites, constraints=restricoes)
    
    pesos_max_sharpe = otimizacao_sharpe.x
    ret_sharpe, vol_sharpe = calcular_performance_portfolio(pesos_max_sharpe, retornos_esperados, cov_matrix)
    
    print("3. A otimizar o Portfólio de Mínima Variância global...")
    otimizacao_min_vol = sco.minimize(minimizar_volatilidade, pesos_iniciais, args=(retornos_esperados, cov_matrix),
                                      method='SLSQP', bounds=limites, constraints=restricoes)
    
    pesos_min_vol = otimizacao_min_vol.x
    ret_min_vol, vol_min_vol = calcular_performance_portfolio(pesos_min_vol, retornos_esperados, cov_matrix)
    
    # 4. Simulação de Monte Carlo para a Nuvem Gráfica
    print(f"4. A gerar {num_simulacoes} carteiras estocásticas (Monte Carlo)...")
    resultados_sim = np.zeros((3, num_simulacoes))
    
    for i in range(num_simulacoes):
        pesos_aleatorios = np.random.random(num_ativos)
        pesos_aleatorios /= np.sum(pesos_aleatorios) # Normaliza para 1
        
        ret_sim, vol_sim = calcular_performance_portfolio(pesos_aleatorios, retornos_esperados, cov_matrix)
        
        resultados_sim[0,i] = vol_sim
        resultados_sim[1,i] = ret_sim
        resultados_sim[2,i] = ret_sim / vol_sim # Sharpe Ratio
        
    # 5. Visualização (Gráfico da Fronteira)
    print("5. A renderizar a Fronteira Eficiente...")
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(resultados_sim[0,:], resultados_sim[1,:], c=resultados_sim[2,:], cmap='YlGnBu', marker='o', s=10, alpha=0.5)
    plt.colorbar(scatter, label='Índice de Sharpe (Prêmio de Risco / Volatilidade)')
    
    # Marcações dos Portfólios Ótimos
    plt.scatter(vol_sharpe, ret_sharpe, marker='*', color='r', s=500, label='Máximo Sharpe (Tangente)')
    plt.scatter(vol_min_vol, ret_min_vol, marker='*', color='g', s=500, label='Mínima Variância')
    
    plt.title('Fronteira Eficiente de Markowitz (Dados Históricos: 2010 - 2025)', fontsize=14)
    plt.xlabel('Risco Anualizado (Volatilidade)', fontsize=12)
    plt.ylabel('Retorno Esperado Anualizado (Prêmio de Risco)', fontsize=12)
    plt.legend(labelspacing=0.8)
    
    caminho_grafico = os.path.join(diretorio_dados, "fronteira_eficiente.png")
    plt.savefig(caminho_grafico, dpi=300, bbox_inches='tight')
    plt.close() # Mantém a memória do notebook limpa
    
    # 6. Exportação dos Pesos Ideais
    df_pesos_ideais = pd.DataFrame({
        'Ativo': ativos,
        'Peso_Max_Sharpe': pesos_max_sharpe,
        'Peso_Min_Volatilidade': pesos_min_vol
    })
    # Removemos os ativos que tiveram peso "zero"
    df_pesos_ideais = df_pesos_ideais[(df_pesos_ideais['Peso_Max_Sharpe'] > 0.001) | (df_pesos_ideais['Peso_Min_Volatilidade'] > 0.001)]
    df_pesos_ideais.set_index('Ativo', inplace=True)
    
    caminho_pesos = os.path.join(diretorio_dados, "pesos_carteiras_otimizadas.csv")
    df_pesos_ideais.to_csv(caminho_pesos)
    
    print("\n=== RESUMO DAS CARTEIRAS OTIMIZADAS ===")
    print(f"--> Carteira Máximo Sharpe: Retorno: {ret_sharpe:.4f} | Risco: {vol_sharpe:.4f} | Sharpe: {ret_sharpe/vol_sharpe:.4f}")
    print(f"--> Carteira Mínima Variância: Retorno: {ret_min_vol:.4f} | Risco: {vol_min_vol:.4f} | Sharpe: {ret_min_vol/vol_min_vol:.4f}")
    print(f"Gráfico salvo em: {caminho_grafico}")
    print(f"Pesos das Carteiras salvos em: {caminho_pesos}")
    print("==================================================================")
    
    return df_pesos_ideais

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

# Executa a otimização num instante
carteiras_otimizadas = otimizar_fronteira_eficiente(pasta_dados, num_simulacoes=50000)

=== INÍCIO DA OTIMIZAÇÃO DE MARKOWITZ (FRONTEIRA EFICIENTE) ===
1. A processar 136 ativos para otimização...
2. A otimizar o Portfólio de Máximo Índice de Sharpe...
3. A otimizar o Portfólio de Mínima Variância global...
4. A gerar 50000 carteiras estocásticas (Monte Carlo)...
5. A renderizar a Fronteira Eficiente...

=== RESUMO DAS CARTEIRAS OTIMIZADAS ===
--> Carteira Máximo Sharpe: Retorno: 0.2011 | Risco: 4.7826 | Sharpe: 0.0421
--> Carteira Mínima Variância: Retorno: -0.8321 | Risco: 0.2839 | Sharpe: -2.9309
Gráfico salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\fronteira_eficiente.png
Pesos das Carteiras salvos em: C:\VSCodeWorkspace\TCC_Escrito\data\pesos_carteiras_otimizadas.csv
